# Finance Copilot — Single-Notebook Kaggle Version

This notebook is self-contained and does **not** depend on local paths, external JSON files, or a repo structure.

It includes:
- demo user profile + demo transactions
- transaction cleaning and categorization
- spending analysis
- affordability / goal planning
- simple investment allocation guidance
- spending forecast (baseline + Prophet fallback)
- lightweight tool router
- basic tests

You can later swap the rule-based `process_query` with Gemma 4 inference on Kaggle.


In [8]:
# Optional: install only if needed on Kaggle
# %pip install -q prophet

import json
import math
from datetime import datetime
from typing import Dict, List, Optional

import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

## 1) Demo data and configuration

In [9]:
USER_PROFILE = {
    "monthly_salary": 80000,
    "fixed_expenses": 25000,
    "current_savings": 120000,
    "monthly_emi": 0,
    "risk_level": "moderate",
    "goal_name": "Laptop",
    "goal_amount": 70000,
    "goal_deadline": "2026-12-01",
    "emergency_fund_target_months": 6
}

CATEGORY_RULES = {
    "zomato": "Food",
    "swiggy": "Food",
    "uber": "Transport",
    "ola": "Transport",
    "amazon": "Shopping",
    "flipkart": "Shopping",
    "netflix": "Entertainment",
    "spotify": "Entertainment",
    "rent": "Housing",
    "electricity": "Utilities",
    "wifi": "Utilities",
    "salary": "Salary",
    "sip": "Investments",
    "mutual fund": "Investments",
    "grocery": "Groceries",
    "blinkit": "Groceries",
    "zepto": "Groceries",
    "bigbasket": "Groceries",
    "pharmacy": "Health",
    "apollo": "Health"
}

demo_transactions = [
    {"date": "2026-01-05", "description": "Salary Credit", "amount": 80000, "type": "income", "category": "Salary"},
    {"date": "2026-01-06", "description": "Rent", "amount": -18000, "type": "expense", "category": "Housing"},
    {"date": "2026-01-08", "description": "Zomato order", "amount": -450, "type": "expense", "category": ""},
    {"date": "2026-01-09", "description": "Uber ride", "amount": -220, "type": "expense", "category": ""},
    {"date": "2026-01-10", "description": "Amazon purchase", "amount": -1800, "type": "expense", "category": ""},
    {"date": "2026-01-12", "description": "Electricity bill", "amount": -2100, "type": "expense", "category": ""},
    {"date": "2026-01-15", "description": "SIP investment", "amount": -5000, "type": "expense", "category": "Investments"},
    {"date": "2026-02-05", "description": "Salary Credit", "amount": 80000, "type": "income", "category": "Salary"},
    {"date": "2026-02-06", "description": "Rent", "amount": -18000, "type": "expense", "category": "Housing"},
    {"date": "2026-02-08", "description": "Swiggy order", "amount": -700, "type": "expense", "category": ""},
    {"date": "2026-02-09", "description": "Ola ride", "amount": -180, "type": "expense", "category": ""},
    {"date": "2026-02-11", "description": "Blinkit grocery", "amount": -2400, "type": "expense", "category": ""},
    {"date": "2026-02-14", "description": "Netflix", "amount": -649, "type": "expense", "category": ""},
    {"date": "2026-02-15", "description": "SIP investment", "amount": -5000, "type": "expense", "category": "Investments"},
    {"date": "2026-03-05", "description": "Salary Credit", "amount": 80000, "type": "income", "category": "Salary"},
    {"date": "2026-03-06", "description": "Rent", "amount": -18000, "type": "expense", "category": "Housing"},
    {"date": "2026-03-08", "description": "Zomato order", "amount": -900, "type": "expense", "category": ""},
    {"date": "2026-03-09", "description": "Uber ride", "amount": -350, "type": "expense", "category": ""},
    {"date": "2026-03-10", "description": "Amazon purchase", "amount": -3200, "type": "expense", "category": ""},
    {"date": "2026-03-12", "description": "Apollo pharmacy", "amount": -850, "type": "expense", "category": ""},
    {"date": "2026-03-15", "description": "SIP investment", "amount": -5000, "type": "expense", "category": "Investments"},
]

raw_df = pd.DataFrame(demo_transactions)
raw_df.head()

,date,description,amount,type,category
0,2026-01-05,Salary Credit,80000,income,Salary
1,2026-01-06,Rent,-18000,expense,Housing
2,2026-01-08,Zomato order,-450,expense,
3,2026-01-09,Uber ride,-220,expense,
4,2026-01-10,Amazon purchase,-1800,expense,


## 2) Data loading, validation, and normalization

In [10]:
REQUIRED_COLUMNS = ["date", "description", "amount", "type", "category"]

def load_transactions(data=None, file_path: Optional[str] = None) -> pd.DataFrame:
    """
    Load transactions either from:
    - a list of dicts / DataFrame passed in memory
    - a CSV file path
    """
    if data is not None:
        if isinstance(data, pd.DataFrame):
            return data.copy()
        return pd.DataFrame(data)
    if file_path is not None:
        return pd.read_csv(file_path)
    raise ValueError("Provide either `data` or `file_path`.")

def validate_schema(df: pd.DataFrame) -> bool:
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return True

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    validate_schema(df)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["description"] = df["description"].fillna("").astype(str).str.strip()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["type"] = df["type"].fillna("").astype(str).str.strip().str.lower()
    df["category"] = df["category"].fillna("").astype(str).str.strip()

    df = df.dropna(subset=["date", "amount"])
    df["type"] = df.apply(
        lambda row: "income" if (row["type"] == "income" or row["amount"] > 0) else "expense",
        axis=1,
    )

    # Ensure signs are consistent
    df.loc[df["type"] == "expense", "amount"] = -df.loc[df["type"] == "expense", "amount"].abs()
    df.loc[df["type"] == "income", "amount"] = df.loc[df["type"] == "income", "amount"].abs()
    return df.reset_index(drop=True)

df = normalize_columns(load_transactions(data=raw_df))
df.head()

,date,description,amount,type,category
0,2026-01-05,Salary Credit,80000,income,Salary
1,2026-01-06,Rent,-18000,expense,Housing
2,2026-01-08,Zomato order,-450,expense,
3,2026-01-09,Uber ride,-220,expense,
4,2026-01-10,Amazon purchase,-1800,expense,


## 3) Categorization

In [11]:
def categorize_by_rules(description: str, rules: Dict[str, str]) -> str:
    desc_lower = str(description).lower()
    for keyword, category in rules.items():
        if keyword in desc_lower:
            return category
    return "Other"

def apply_categorization(df: pd.DataFrame, rules: Dict[str, str]) -> pd.DataFrame:
    out = df.copy()
    out["category"] = out.apply(
        lambda row: row["category"] if row["category"] else categorize_by_rules(row["description"], rules),
        axis=1,
    )
    return out

df = apply_categorization(df, CATEGORY_RULES)
df.head()

,date,description,amount,type,category
0,2026-01-05,Salary Credit,80000,income,Salary
1,2026-01-06,Rent,-18000,expense,Housing
2,2026-01-08,Zomato order,-450,expense,Food
3,2026-01-09,Uber ride,-220,expense,Transport
4,2026-01-10,Amazon purchase,-1800,expense,Shopping


## 4) Core analysis functions

In [12]:
def filter_month(df: pd.DataFrame, month: Optional[str] = None) -> pd.DataFrame:
    if month is None:
        return df.copy()
    return df[df["date"].dt.strftime("%Y-%m") == month].copy()

def get_total_income(df: pd.DataFrame, month: Optional[str] = None) -> float:
    temp = filter_month(df, month)
    return float(temp.loc[temp["type"] == "income", "amount"].sum())

def get_total_expenses(df: pd.DataFrame, month: Optional[str] = None) -> float:
    temp = filter_month(df, month)
    return float(temp.loc[temp["type"] == "expense", "amount"].abs().sum())

def get_category_breakdown(df: pd.DataFrame, month: Optional[str] = None) -> Dict[str, float]:
    temp = filter_month(df, month)
    temp = temp[temp["type"] == "expense"].copy()
    if temp.empty:
        return {}
    breakdown = temp.groupby("category")["amount"].sum().abs().sort_values(ascending=False)
    return {k: float(v) for k, v in breakdown.items()}

def get_top_expense_categories(df: pd.DataFrame, n: int = 5, month: Optional[str] = None) -> Dict[str, float]:
    breakdown = get_category_breakdown(df, month)
    items = list(breakdown.items())[:n]
    return dict(items)

def get_monthly_surplus(df: pd.DataFrame, monthly_salary: float, fixed_expenses: float = 0, month: Optional[str] = None) -> float:
    variable_expenses = get_total_expenses(df, month)
    return float(monthly_salary - fixed_expenses - variable_expenses)

def detect_overspending(df: pd.DataFrame, threshold: float = 1.2) -> List[Dict]:
    expenses = df[df["type"] == "expense"].copy()
    if expenses.empty:
        return []

    expenses["month"] = expenses["date"].dt.strftime("%Y-%m")
    monthly_cat = (
        expenses.groupby(["month", "category"])["amount"]
        .sum()
        .abs()
        .reset_index()
        .sort_values("month")
    )

    latest_month = monthly_cat["month"].max()
    previous = monthly_cat[monthly_cat["month"] < latest_month]
    current = monthly_cat[monthly_cat["month"] == latest_month]

    flags = []
    for _, row in current.iterrows():
        prev_cat = previous[previous["category"] == row["category"]]["amount"]
        if len(prev_cat) == 0:
            continue
        baseline = prev_cat.mean()
        if baseline > 0 and row["amount"] > threshold * baseline:
            flags.append({
                "month": latest_month,
                "category": row["category"],
                "current_spend": float(row["amount"]),
                "baseline": float(baseline),
                "increase_pct": float((row["amount"] - baseline) / baseline * 100)
            })
    return flags

print("Total expenses:", get_total_expenses(df))
print("Category breakdown:", get_category_breakdown(df))
print("Monthly surplus:", get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"]))
print("Overspending flags:", detect_overspending(df))

Total expenses: 82799.0
Category breakdown: {'Housing': 54000.0, 'Investments': 15000.0, 'Shopping': 5000.0, 'Groceries': 2400.0, 'Utilities': 2100.0, 'Food': 2050.0, 'Health': 850.0, 'Transport': 750.0, 'Entertainment': 649.0}
Monthly surplus: -27799.0
Overspending flags: [{'month': '2026-03', 'category': 'Food', 'current_spend': 900.0, 'baseline': 575.0, 'increase_pct': 56.52173913043478}, {'month': '2026-03', 'category': 'Shopping', 'current_spend': 3200.0, 'baseline': 1800.0, 'increase_pct': 77.77777777777779}, {'month': '2026-03', 'category': 'Transport', 'current_spend': 350.0, 'baseline': 200.0, 'increase_pct': 75.0}]


## 5) Forecasting

In [13]:
def prepare_monthly_series(df: pd.DataFrame, category: Optional[str] = None) -> pd.Series:
    temp = df[df["type"] == "expense"].copy()
    if category is not None:
        temp = temp[temp["category"] == category].copy()

    if temp.empty:
        return pd.Series(dtype=float)

    monthly = temp.groupby(temp["date"].dt.to_period("M"))["amount"].sum().abs().sort_index()
    monthly.index = monthly.index.to_timestamp()
    monthly.name = "y"
    return monthly

def forecast_with_baseline(series: pd.Series, periods: int = 3) -> List[float]:
    if series.empty:
        return [0.0] * periods
    window = min(3, len(series))
    baseline = float(series.tail(window).mean())
    return [baseline] * periods

def forecast_with_prophet(series: pd.Series, periods: int = 3) -> List[float]:
    if series.empty:
        return [0.0] * periods
    try:
        from prophet import Prophet

        df_prophet = pd.DataFrame({"ds": series.index, "y": series.values})
        model = Prophet(daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=True)
        model.fit(df_prophet)

        future = model.make_future_dataframe(periods=periods, freq="MS")
        forecast = model.predict(future)
        preds = forecast["yhat"].tail(periods).clip(lower=0)
        return [float(x) for x in preds.tolist()]
    except Exception as e:
        print(f"Prophet unavailable or failed ({e}); using baseline instead.")
        return forecast_with_baseline(series, periods)

def evaluate_forecast(actual: List[float], predicted: List[float]) -> Dict[str, float]:
    if len(actual) == 0 or len(actual) != len(predicted):
        return {"mape": float("nan")}
    return {"mape": float(mean_absolute_percentage_error(actual, predicted))}

monthly_series = prepare_monthly_series(df)
baseline_forecast = forecast_with_baseline(monthly_series, periods=3)
monthly_series, baseline_forecast

(date
 2026-01-01    27570
 2026-02-01    26929
 2026-03-01    28300
 Freq: MS, Name: y, dtype: int64,
 [27599.666666666668, 27599.666666666668, 27599.666666666668])

## 6) Planning and investing logic

In [14]:
def calculate_months_left(deadline: str, today: Optional[datetime] = None) -> int:
    today = today or datetime.now()
    deadline_dt = datetime.strptime(deadline, "%Y-%m-%d")
    months = (deadline_dt.year - today.year) * 12 + (deadline_dt.month - today.month)
    return max(1, months)

def affordability_check(cost: float, monthly_surplus: float, current_savings: float, min_buffer: float = 50000) -> Dict:
    post_purchase_savings = current_savings - cost
    affordable_now = post_purchase_savings >= min_buffer

    return {
        "affordable": bool(affordable_now),
        "cost": float(cost),
        "monthly_surplus": float(monthly_surplus),
        "current_savings": float(current_savings),
        "post_purchase_savings": float(post_purchase_savings),
        "recommended_min_buffer": float(min_buffer),
        "message": (
            "Purchase looks affordable."
            if affordable_now
            else "Purchase would reduce savings below your minimum buffer."
        ),
    }

def goal_check(goal_amount: float, current_savings: float, deadline: str, monthly_surplus: float, today: Optional[datetime] = None) -> Dict:
    months_left = calculate_months_left(deadline, today=today)
    remaining = max(0.0, goal_amount - current_savings)
    required_monthly_savings = remaining / months_left if months_left > 0 else remaining
    gap_vs_surplus = required_monthly_savings - monthly_surplus
    status = "on_track" if monthly_surplus >= required_monthly_savings else "behind"

    return {
        "goal_amount": float(goal_amount),
        "current_savings": float(current_savings),
        "deadline": deadline,
        "months_left": int(months_left),
        "amount_remaining": float(remaining),
        "required_monthly_savings": float(required_monthly_savings),
        "current_monthly_surplus": float(monthly_surplus),
        "status": status,
        "gap_vs_surplus": float(gap_vs_surplus),
        "message": (
            "You are on track for this goal."
            if status == "on_track"
            else f"You need about ₹{gap_vs_surplus:,.0f} more surplus per month."
        ),
    }

def suggest_cuts(df: pd.DataFrame, target_extra_savings: float, top_n: int = 3) -> Dict:
    non_essential = {"Food", "Shopping", "Entertainment", "Transport"}
    breakdown = get_category_breakdown(df)
    ranked = [(cat, amt) for cat, amt in breakdown.items() if cat in non_essential]
    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)[:top_n]

    if not ranked:
        return {"target_extra_savings": float(target_extra_savings), "suggested_cuts": []}

    total_ranked = sum(amt for _, amt in ranked)
    suggestions = []
    for cat, amt in ranked:
        share = amt / total_ranked if total_ranked else 0
        suggestions.append({
            "category": cat,
            "current_spend": float(amt),
            "suggested_cut": float(round(target_extra_savings * share, 2))
        })
    return {"target_extra_savings": float(target_extra_savings), "suggested_cuts": suggestions}

def estimate_investable_surplus(monthly_surplus: float, emergency_fund_gap: float) -> float:
    if monthly_surplus <= 0:
        return 0.0
    reserve_contribution = max(0.0, emergency_fund_gap / 12)
    return float(max(0.0, monthly_surplus - reserve_contribution))

def recommend_allocation(monthly_surplus: float, risk_level: str, goal_horizon_years: int) -> Dict:
    risk_level = str(risk_level).lower().strip()

    if monthly_surplus <= 0:
        return {
            "monthly_surplus": float(monthly_surplus),
            "allocation_pct": {"equity": 0.0, "debt": 0.0, "cash": 1.0},
            "allocation_amount": {"equity": 0.0, "debt": 0.0, "cash": 0.0},
            "message": "No investable surplus right now."
        }

    if risk_level == "low":
        alloc = {"equity": 0.2, "debt": 0.5, "cash": 0.3}
    elif risk_level == "high":
        alloc = {"equity": 0.7, "debt": 0.2, "cash": 0.1}
    else:
        alloc = {"equity": 0.5, "debt": 0.35, "cash": 0.15}

    if goal_horizon_years < 3:
        alloc["cash"] += 0.15
        alloc["equity"] = max(0.0, alloc["equity"] - 0.15)

    total = sum(alloc.values())
    alloc = {k: v / total for k, v in alloc.items()}
    alloc_amount = {k: float(monthly_surplus * v) for k, v in alloc.items()}

    return {
        "monthly_surplus": float(monthly_surplus),
        "allocation_pct": alloc,
        "allocation_amount": alloc_amount,
        "message": "Conservative rule-based allocation suggestion."
    }

monthly_surplus = get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"])
goal_result = goal_check(USER_PROFILE["goal_amount"], USER_PROFILE["current_savings"], USER_PROFILE["goal_deadline"], monthly_surplus)
cut_result = suggest_cuts(df, 5000)
allocation_result = recommend_allocation(monthly_surplus, USER_PROFILE["risk_level"], goal_horizon_years=5)

goal_result, cut_result, allocation_result

({'goal_amount': 70000.0,
  'current_savings': 120000.0,
  'deadline': '2026-12-01',
  'months_left': 8,
  'amount_remaining': 0.0,
  'required_monthly_savings': 0.0,
  'current_monthly_surplus': -27799.0,
  'status': 'behind',
  'gap_vs_surplus': 27799.0,
  'message': 'You need about ₹27,799 more surplus per month.'},
 {'target_extra_savings': 5000.0,
  'suggested_cuts': [{'category': 'Shopping',
    'current_spend': 5000.0,
    'suggested_cut': 3205.13},
   {'category': 'Food', 'current_spend': 2050.0, 'suggested_cut': 1314.1},
   {'category': 'Transport',
    'current_spend': 750.0,
    'suggested_cut': 480.77}]},
 {'monthly_surplus': -27799.0,
  'allocation_pct': {'equity': 0.0, 'debt': 0.0, 'cash': 1.0},
  'allocation_amount': {'equity': 0.0, 'debt': 0.0, 'cash': 0.0},
  'message': 'No investable surplus right now.'})

## 7) Tool layer

In [15]:
def format_currency(amount: float) -> str:
    return f"₹{amount:,.0f}"

def analyze_spending(month_range: Optional[str] = None) -> Dict:
    return {
        "month": month_range,
        "total_income": get_total_income(df, month_range),
        "total_expenses": get_total_expenses(df, month_range),
        "category_breakdown": get_category_breakdown(df, month_range),
        "top_expense_categories": get_top_expense_categories(df, 5, month_range),
        "overspending_flags": detect_overspending(filter_month(df, month_range) if month_range else df),
    }

def category_breakdown_tool(month_range: Optional[str] = None) -> Dict:
    return {"month": month_range, "category_breakdown": get_category_breakdown(df, month_range)}

def affordability_check_tool(item_cost: float, min_buffer: float = 50000) -> Dict:
    monthly_surplus = get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"])
    return affordability_check(item_cost, monthly_surplus, USER_PROFILE["current_savings"], min_buffer=min_buffer)

def goal_check_tool(goal_amount: Optional[float] = None, deadline: Optional[str] = None) -> Dict:
    monthly_surplus = get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"])
    return goal_check(
        goal_amount=goal_amount if goal_amount is not None else USER_PROFILE["goal_amount"],
        current_savings=USER_PROFILE["current_savings"],
        deadline=deadline if deadline is not None else USER_PROFILE["goal_deadline"],
        monthly_surplus=monthly_surplus,
    )

def suggest_cuts_tool(target_extra_savings: float) -> Dict:
    return suggest_cuts(df, target_extra_savings)

def recommend_allocation_tool() -> Dict:
    monthly_surplus = get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"])
    return recommend_allocation(
        monthly_surplus=monthly_surplus,
        risk_level=USER_PROFILE["risk_level"],
        goal_horizon_years=5,
    )

def forecast_spending(category: Optional[str] = None, months: int = 3, method: str = "baseline") -> Dict:
    series = prepare_monthly_series(df, category)
    if method == "prophet":
        preds = forecast_with_prophet(series, periods=months)
    else:
        preds = forecast_with_baseline(series, periods=months)

    future_months = pd.date_range(
        start=(series.index.max() + pd.offsets.MonthBegin(1)) if len(series) else pd.Timestamp.today().normalize(),
        periods=months,
        freq="MS",
    ).strftime("%Y-%m").tolist()

    return {
        "category": category,
        "method": method,
        "history": {idx.strftime("%Y-%m"): float(val) for idx, val in series.items()},
        "forecast": dict(zip(future_months, [float(x) for x in preds])),
    }

TOOLS = {
    "analyze_spending": analyze_spending,
    "category_breakdown": category_breakdown_tool,
    "affordability_check": affordability_check_tool,
    "goal_check": goal_check_tool,
    "suggest_cuts": suggest_cuts_tool,
    "recommend_allocation": recommend_allocation_tool,
    "forecast_spending": forecast_spending,
}

def call_tool(tool_name: str, **kwargs) -> Dict:
    if tool_name not in TOOLS:
        return {"error": f"Tool '{tool_name}' not found."}
    try:
        return TOOLS[tool_name](**kwargs)
    except Exception as e:
        return {"error": str(e)}

call_tool("analyze_spending")

{'month': None,
 'total_income': 240000.0,
 'total_expenses': 82799.0,
 'category_breakdown': {'Housing': 54000.0,
  'Investments': 15000.0,
  'Shopping': 5000.0,
  'Groceries': 2400.0,
  'Utilities': 2100.0,
  'Food': 2050.0,
  'Health': 850.0,
  'Transport': 750.0,
  'Entertainment': 649.0},
 'top_expense_categories': {'Housing': 54000.0,
  'Investments': 15000.0,
  'Shopping': 5000.0,
  'Groceries': 2400.0,
  'Utilities': 2100.0},
 'overspending_flags': [{'month': '2026-03',
   'category': 'Food',
   'current_spend': 900.0,
   'baseline': 575.0,
   'increase_pct': 56.52173913043478},
  {'month': '2026-03',
   'category': 'Shopping',
   'current_spend': 3200.0,
   'baseline': 1800.0,
   'increase_pct': 77.77777777777779},
  {'month': '2026-03',
   'category': 'Transport',
   'current_spend': 350.0,
   'baseline': 200.0,
   'increase_pct': 75.0}]}

## 8) Lightweight query router (replace later with Gemma 4)

In [16]:
def process_query(query: str) -> Dict:
    q = query.lower().strip()

    if "afford" in q or "buy" in q or "purchase" in q:
        return {
            "selected_tool": "affordability_check",
            "result": affordability_check_tool(item_cost=40000)
        }
    if "save" in q or "goal" in q:
        return {
            "selected_tool": "goal_check",
            "result": goal_check_tool()
        }
    if "cut" in q or "reduce" in q or "overspend" in q:
        return {
            "selected_tool": "suggest_cuts",
            "result": suggest_cuts_tool(target_extra_savings=5000)
        }
    if "invest" in q or "allocate" in q:
        return {
            "selected_tool": "recommend_allocation",
            "result": recommend_allocation_tool()
        }
    if "forecast" in q or "next month" in q:
        return {
            "selected_tool": "forecast_spending",
            "result": forecast_spending()
        }
    if "category" in q or "where" in q or "spend" in q or "expense" in q:
        return {
            "selected_tool": "analyze_spending",
            "result": analyze_spending()
        }

    return {
        "selected_tool": None,
        "result": {"message": "Sorry, I could not map this query to a tool yet."}
    }

sample_queries = [
    "Where am I overspending?",
    "Can I afford a 40k phone?",
    "How do I save for my goal?",
    "How should I allocate my money?",
    "Forecast my spending"
]

for q in sample_queries:
    print("\nQuery:", q)
    print(json.dumps(process_query(q), indent=2))


Query: Where am I overspending?
{
  "selected_tool": "suggest_cuts",
  "result": {
    "target_extra_savings": 5000.0,
    "suggested_cuts": [
      {
        "category": "Shopping",
        "current_spend": 5000.0,
        "suggested_cut": 3205.13
      },
      {
        "category": "Food",
        "current_spend": 2050.0,
        "suggested_cut": 1314.1
      },
      {
        "category": "Transport",
        "current_spend": 750.0,
        "suggested_cut": 480.77
      }
    ]
  }
}

Query: Can I afford a 40k phone?
{
  "selected_tool": "affordability_check",
  "result": {
    "affordable": true,
    "cost": 40000.0,
    "monthly_surplus": -27799.0,
    "current_savings": 120000.0,
    "post_purchase_savings": 80000.0,
    "recommended_min_buffer": 50000.0,
    "message": "Purchase looks affordable."
  }
}

Query: How do I save for my goal?
{
  "selected_tool": "goal_check",
  "result": {
    "goal_amount": 70000.0,
    "current_savings": 120000.0,
    "deadline": "2026-12-01",
 

## 9) Gemma 4 hook (optional)

In [17]:
# Replace the rule-based `process_query` with Gemma 4 later.
# Example sketch only — model name / access may differ on Kaggle.

GEMMA_ROUTER_PROMPT = '''
You are a finance tool router.
Available tools:
- analyze_spending
- category_breakdown
- affordability_check
- goal_check
- suggest_cuts
- recommend_allocation
- forecast_spending

Return only JSON with:
{"tool_name": "...", "arguments": {...}}
'''

print("Gemma integration placeholder ready. Plug model inference here when your Kaggle Gemma setup is ready.")

Gemma integration placeholder ready. Plug model inference here when your Kaggle Gemma setup is ready.


## 10) Basic tests

In [18]:
def test_analysis():
    total_exp = get_total_expenses(df)
    assert total_exp > 0, "Total expenses should be positive"

    breakdown = get_category_breakdown(df)
    assert isinstance(breakdown, dict), "Breakdown should be a dict"
    assert "Housing" in breakdown, "Expected known category missing"

    surplus = get_monthly_surplus(df, USER_PROFILE["monthly_salary"], USER_PROFILE["fixed_expenses"])
    assert isinstance(surplus, float), "Surplus should be numeric"

def test_planner():
    result = affordability_check(40000, 20000, 120000, 50000)
    assert result["affordable"] is True, "Should be affordable"

    result = affordability_check(80000, 20000, 120000, 50000)
    assert result["affordable"] is False, "Should not be affordable"

    result = goal_check(200000, 50000, "2026-12-01", 20000, today=datetime(2026, 4, 1))
    assert result["status"] in ["on_track", "behind"], "Status should be valid"

def test_tools():
    result = analyze_spending()
    assert "total_expenses" in result

    result = affordability_check_tool(40000)
    assert "affordable" in result

    result = forecast_spending()
    assert "forecast" in result

test_analysis()
test_planner()
test_tools()
print("All tests passed.")

All tests passed.


## 11) Simple notebook demo

In [19]:
print("Finance Copilot Notebook Demo")
print("-" * 40)
print("User profile:")
print(json.dumps(USER_PROFILE, indent=2))

print("\nSpending summary:")
print(json.dumps(analyze_spending(), indent=2))

print("\nGoal status:")
print(json.dumps(goal_check_tool(), indent=2))

print("\nAllocation suggestion:")
print(json.dumps(recommend_allocation_tool(), indent=2))

print("\nSpending forecast:")
print(json.dumps(forecast_spending(), indent=2))

Finance Copilot Notebook Demo
----------------------------------------
User profile:
{
  "monthly_salary": 80000,
  "fixed_expenses": 25000,
  "current_savings": 120000,
  "monthly_emi": 0,
  "risk_level": "moderate",
  "goal_name": "Laptop",
  "goal_amount": 70000,
  "goal_deadline": "2026-12-01",
  "emergency_fund_target_months": 6
}

Spending summary:
{
  "month": null,
  "total_income": 240000.0,
  "total_expenses": 82799.0,
  "category_breakdown": {
    "Housing": 54000.0,
    "Investments": 15000.0,
    "Shopping": 5000.0,
    "Groceries": 2400.0,
    "Utilities": 2100.0,
    "Food": 2050.0,
    "Health": 850.0,
    "Transport": 750.0,
    "Entertainment": 649.0
  },
  "top_expense_categories": {
    "Housing": 54000.0,
    "Investments": 15000.0,
    "Shopping": 5000.0,
    "Groceries": 2400.0,
    "Utilities": 2100.0
  },
  "overspending_flags": [
    {
      "month": "2026-03",
      "category": "Food",
      "current_spend": 900.0,
      "baseline": 575.0,
      "increase_pct